In [ ]:
#This script uses the Google Maps Places API (Nearby Search) to find LPG-related points of interest across Nigeria. 
#All coordinates are in WGS84 (EPSG:4326) reference system, with latitude and longitude in decimal degrees. Results are saved in CSV, JSON, gpkg formats.

In [2]:
# Cell 1: Setup and configuration
import requests
import time
import csv
import json
from datetime import datetime

API_KEY = "key"  # Replace with your actual API key

# Nigeria bounding box
NIGERIA_BBOX = {
    'min_lat': 4.0,
    'max_lat': 14.0,
    'min_lon': 2.5,
    'max_lon': 14.5
}

STEP = 0.65  # degrees, ~72 km – ensures circle overlap (radius 50 km)
RADIUS = 50000  # meters

# Keywords (balanced for coverage vs false positives)
KEYWORDS = [
    "LPG refill station",
    "gas station with LPG",
    "LPG filling point",
    "LPG dealer",
    "gas cylinder exchange",
    "cooking gas",
    "LPG"
    #"gas plant", #to avoid false positive is better to turn of
    #"LPG bulk plant", #to avoid false positive is better to turn of (it is not a retail point, but "all'ingrosso")     
]

In [3]:
# Cell 2: Generate grid of search centers
def generate_grid(bbox, step):
    points = []
    lat = bbox['min_lat']
    while lat <= bbox['max_lat']:
        lon = bbox['min_lon']
        while lon <= bbox['max_lon']:
            points.append((round(lat, 4), round(lon, 4)))
            lon += step
        lat += step
    return points

grid = generate_grid(NIGERIA_BBOX, STEP)
print(f"Grid points: {len(grid)}")

Grid points: 304


In [4]:
# Cell 3: Nearby search function with pagination
def nearby_search(lat, lon, keyword, radius=50000):
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    results = []
    params = {
        'location': f"{lat},{lon}",
        'radius': radius,
        'keyword': keyword,
        'key': API_KEY
    }

    while True:
        resp = requests.get(url, params=params)
        data = resp.json()

        if data['status'] not in ['OK', 'ZERO_RESULTS']:
            print(f"API error: {data['status']} - {data.get('error_message', '')}")
            break

        for place in data.get('results', []):
            results.append({
                'place_id': place['place_id'],
                'name': place.get('name', ''),
                'address': place.get('vicinity', ''),   # short address
                'lat': place['geometry']['location']['lat'],
                'lng': place['geometry']['location']['lng'],
                'types': ', '.join(place.get('types', [])),
                'keyword': keyword,
                'search_lat': lat,
                'search_lon': lon,
                'rating': place.get('rating'),        
                'user_ratings_total': place.get('user_ratings_total', 0)  
            })

        if 'next_page_token' in data:
            params['pagetoken'] = data['next_page_token']
            time.sleep(2)   # token needs time to become valid
        else:
            break

    return results

In [5]:
# # Cella 4: Test only on Lagos (output ridotto)
# all_places = {}
# total_api_calls = 0

# # Coordinate di Lagos (centro)
# test_points = [(6.5244, 3.3792)]  

# for idx, (lat, lon) in enumerate(test_points):
#     print(f"Test point {idx+1}: ({lat}, {lon})")
#     point_new = 0
#     for kw in KEYWORDS:
#         try:
#             places = nearby_search(lat, lon, kw)
#             total_api_calls += 1 + (len(places) // 60)
#             for p in places:
#                 if p['place_id'] not in all_places:
#                     all_places[p['place_id']] = p
#                     point_new += 1
#         except Exception as e:
#             print(f"  Error with '{kw}': {e}")
#         time.sleep(0.2)
#     print(f"  Found {point_new} new places in this point\n")

# print(f"\nTest completed. Estimated API calls: {total_api_calls}")
# print(f"Unique places found in Lagos: {len(all_places)}")

# # Mostra un anteprima dei risultati
# if all_places:
#     print("\nFirst 5 results:")
#     for i, (place_id, place) in enumerate(list(all_places.items())[:5]):
#         print(f"  {i+1}. {place['name']} - ({place['lat']}, {place['lng']})")
# else:
#     print("No results found. Check your API key and keywords.")

In [6]:
# Cell 4: Main loop with error monitoring
all_places = {}
total_api_calls = 0
error_count = 0
first_error = None

for idx, (lat, lon) in enumerate(grid):
    if (idx + 1) % 10 == 0 or idx == 0:
        print(f"Processing point {idx+1}/{len(grid)}: ({lat}, {lon})")
    
    for kw in KEYWORDS:
        try:
            places = nearby_search(lat, lon, kw)
            total_api_calls += 1 + (len(places) // 60)
            for p in places:
                if p['place_id'] not in all_places:
                    all_places[p['place_id']] = p
        except Exception as e:
            error_count += 1
            if first_error is None:
                first_error = f"{kw} at ({lat},{lon}): {e}"
                print(f"  ⚠️ First error: {first_error}")

        time.sleep(0.2)

    if (idx + 1) % 10 == 0:
        with open('lpg_nigeria_interim.json', 'w') as f:
            json.dump(list(all_places.values()), f, indent=2)
        print(f"  Checkpoint: {len(all_places)} unique, {error_count} errors")

print(f"\nDone. Estimated API calls: {total_api_calls}")
print(f"Unique places found: {len(all_places)}")
print(f"Total errors: {error_count}")
if error_count > 0:
    print(f"Example error: {first_error}")

Processing point 1/304: (4.0, 2.5)
Processing point 10/304: (4.0, 8.35)
  Checkpoint: 0 unique, 0 errors
Processing point 20/304: (4.65, 2.5)
  Checkpoint: 51 unique, 0 errors
Processing point 30/304: (4.65, 9.0)
  Checkpoint: 370 unique, 0 errors
Processing point 40/304: (5.3, 3.15)
  Checkpoint: 374 unique, 0 errors
Processing point 50/304: (5.3, 9.65)
  Checkpoint: 585 unique, 0 errors
Processing point 60/304: (5.95, 3.8)
  Checkpoint: 592 unique, 0 errors
Processing point 70/304: (5.95, 10.3)
  Checkpoint: 876 unique, 0 errors
Processing point 80/304: (6.6, 4.45)
  Checkpoint: 1165 unique, 0 errors
Processing point 90/304: (6.6, 10.95)
  Checkpoint: 1357 unique, 0 errors
Processing point 100/304: (7.25, 5.1)
  Checkpoint: 1801 unique, 0 errors
Processing point 110/304: (7.25, 11.6)
  Checkpoint: 1947 unique, 0 errors
Processing point 120/304: (7.9, 5.75)
  Checkpoint: 2124 unique, 0 errors
Processing point 130/304: (7.9, 12.25)
  Checkpoint: 2209 unique, 0 errors
Processing point 1

In [7]:
# Cell 5: Save final results to CSV (and JSON)
results = list(all_places.values())
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

csv_file = f"lpg_nigeria_{timestamp}.csv"
with open(csv_file, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['place_id', 'name', 'address', 'lat', 'lng', 'types', 'keyword', 'search_lat', 'search_lon', 'rating', 'user_ratings_total']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

json_file = csv_file.replace('.csv', '.json')
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved {len(results)} records:")
print(f"  CSV: {csv_file}")
print(f"  JSON: {json_file}")

Saved 3812 records:
  CSV: lpg_nigeria_20260316_104544.csv
  JSON: lpg_nigeria_20260316_104544.json


In [17]:
# Cell 6: Quick stats
from collections import Counter

print("Summary by keyword:")
kw_counter = Counter([r['keyword'] for r in results])
for kw, cnt in kw_counter.most_common():
    print(f"  {kw}: {cnt}")

print("\nFirst 5 records:")
for r in results[:5]:
    print(f"  {r['name']} - ({r['lat']}, {r['lng']})")

total_places = len(results)

# Places with at least one review
with_reviews = [r for r in results if r.get('user_ratings_total', 0) > 0]
num_with_reviews = len(with_reviews)
print(f"\n--- Review Statistics ---")
print(f"Total places: {total_places}")
print(f"Places with at least one review: {num_with_reviews} ({num_with_reviews/total_places*100:.1f}%)")

if with_reviews:
    # Rating categories
    below_3 = [r for r in with_reviews if r.get('rating', 0) < 3]
    print(f"Places with average rating < 3: {len(below_3)}")
    below_2_5 = [r for r in with_reviews if r.get('rating', 0) < 2.5] #selected for the huff
    print(f"Places with average rating < 2,5: {len(below_2_5)}")
    below_2 = [r for r in with_reviews if r.get('rating', 0) < 2]
    print(f"Places with average rating < 2: {len(below_2)}")
    
    # Review count thresholds
    less_than_4_reviews = [r for r in with_reviews if r.get('user_ratings_total', 0) < 4.1]
    less_than_3_reviews = [r for r in with_reviews if r.get('user_ratings_total', 0) < 3.1] 
    less_than_2_reviews = [r for r in with_reviews if r.get('user_ratings_total', 0) < 2.1]
    print(f"Places with fewer or equal to 4 reviews (but more than 0): {len(less_than_4_reviews)}")
    print(f"Places with fewer or equal to 3 reviews (but more than 0): {len(less_than_3_reviews)}") #selected for the huff
    print(f"Places with fewer or equal to 2 reviews (but more than 0): {len(less_than_2_reviews)}")
else:
    print("No places with reviews found.")


Summary by keyword:
  gas station with LPG: 1660
  cooking gas: 1026
  LPG refill station: 739
  LPG filling point: 163
  gas cylinder exchange: 132
  LPG: 86
  LPG dealer: 6

First 5 records:
  BOCOM Petrol Station, Muea - (4.1693647, 9.301529)
  MUWA GAS DELIVERY SERVICE - (4.1559658, 9.2632243)
  MRS Petrol Station - (4.1532496, 9.2419495)
  TotalEnergies BUEA - (4.1520829, 9.2941899)
  VEC Gas and Kitchen Essentials - (4.1623498, 9.2587777)

--- Review Statistics ---
Total places: 3812
Places with at least one review: 3050 (80.0%)
Places with average rating < 3: 111
Places with average rating < 2,5: 64
Places with average rating < 2: 33
Places with fewer or equal to 4 reviews (but more than 0): 1296
Places with fewer or equal to 3 reviews (but more than 0): 1147
Places with fewer or equal to 2 reviews (but more than 0): 877


In [12]:
# Cell 8: Create GeoPackage with LPG supply points in EPSG:3857
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# Convert the list of dicts to a DataFrame
df = pd.DataFrame(results)

# Create Point geometry from longitude and latitude
geometry = [Point(lon, lat) for lon, lat in zip(df['lng'], df['lat'])]

# Build GeoDataFrame in WGS84 first
gdf = gpd.GeoDataFrame(
    df[['place_id', 'name', 'address', 'types', 'keyword', 'search_lat', 'search_lon', 'rating', 'user_ratings_total']],
    geometry=geometry,
    crs="EPSG:4326"
)

# Add original coordinate columns (optional)
gdf['lat'] = df['lat']
gdf['lon'] = df['lng']

# Reproject to EPSG:3857 (Web Mercator)
gdf_3857 = gdf.to_crs("EPSG:3857")

# Save as GeoPackage
output_gpkg = 'lpg_points_maps_nigeria_3857.gpkg'
gdf_3857.to_file(output_gpkg, driver='GPKG', layer='lpg_points')

print(f"   ✅ GeoPackage saved as: {output_gpkg}")


   ✅ GeoPackage saved as: lpg_points_maps_nigeria_3857.gpkg
